# `dde` package demo

`dde` fits Deep Discrete Encoder (DDE) models: a hierarchy of $D$ binary latent layers
$A_D \to A_{D-1} \to \cdots \to A_1 \to Y$, where each layer is a sparse logistic
model of the layer above and the observed layer $Y$ follows a model-specific
distribution. A fitted model returns, for every layer, a coefficient matrix `B_list[d]`
(column 0 is the intercept, columns 1.. are the loadings on the parent layer), the
top-layer probabilities `prop`, the latent states `A_list`, and, for the Normal DDE and
SemiDDE, the residual variances `gamma`.

This notebook walks through the public interface on simulated data:

1. the models and the two ways of choosing the latent widths;
2. simulating a small Normal DDE;
3. `fit()` with fixed latent widths;
4. `fit()` with widths chosen by cumulative shrinkage (CSP);
5. `select_hyperparameters()` on an explicit grid and on the automatic grid;
6. the other models (Bernoulli, Poisson, SemiDDE, CopDDE);
7. plots of the estimated weights and the anchor variables.

Install the package with the plotting extra from the repository root
(`pip install -e ".[viz]"`) and run the notebook from the `examples/` directory.
It takes a few minutes.

## 1. Models and latent widths

Every model is reached through the same call,
`fit(Y, model=..., family=..., D=..., dimension_method=..., ...)`.
`model` picks the formulation, `family` the observed-data distribution (`model="DDE"` only),
and `dimension_method` whether the latent widths are supplied (`"fixed"`, pass `K_list`)
or chosen by the cumulative shrinkage process (`"csp"`, omit `K_list`).

| `model` | `family` | data | fixed widths | CSP |
|---|---|---|---|---|
| `"DDE"` | `"Normal"` | continuous | penalized SAEM | CUSP shrinkage from the maximal widths |
| `"DDE"` | `"Bernoulli"` | binary 0/1 | penalized SAEM | CUSP shrinkage |
| `"DDE"` | `"Poisson"` | counts | penalized SAEM | CUSP shrinkage |
| `"SemiDDE"` | – | continuous, unknown monotone margins | SAEM alternated with the transformation estimate | CUSP shrinkage with the transformation estimate |
| `"CopDDE"` | – | continuous, discrete, ordinal or mixed | rank-likelihood SAEM | rank-likelihood CUSP shrinkage |

The main hyperparameters (defaults in brackets; omitted ones are filled in and recorded
in `result.diagnostics["default_arguments"]`):

* `K_list`: latent widths from the shallowest layer to the deepest, e.g. `[6, 2]`. Fixed
  widths require it; CSP starts from the maximal identifiable widths
  `floor(J/3), floor(floor(J/3)/3), ...` and shrinks them.
* `pen_list`: truncated-lasso penalty of each layer (fixed widths) [$N^{1/4}$ in every layer].
* `tau`: hard threshold on the loadings; a loading below `tau` in absolute value is set to
  zero and defines the recovered graph [0.3; CopDDE $\max(0.3, 3N^{-0.3})$].
* `t_spike`: spike scale of each layer (CSP) [(0.03, 0.05) for $D=2$, (0.01, 0.03, 0.005) for $D=3$].
* `temp`: initial annealing temperature (CSP) [0.7 for $D \le 2$, 0.9 otherwise].
* `tol`: tolerance on the relative change of the parameters [0.005];
  `max_it`: iteration limit [50; Bernoulli and Poisson CSP run 100 iterations];
  `C`: latent draws per iteration [1].

In [1]:
import time
import numpy as np
import pandas as pd
from scipy.optimize import linear_sum_assignment

from dde import fit, select_hyperparameters
from dde.identifiability import build_true_params
from dde.simulation import (generate_X_normal, generate_X_bernoulli, generate_X_poisson,
                            generate_X_Cop_marg_emp_D)

np.set_printoptions(precision=3, suppress=True)
SEED = 2026

## 2. Simulated data

`build_true_params(K_list, J, max_val)` builds the sparse, identifiable loading matrices
used in the package's simulation studies (each latent variable has anchor observed
variables plus a few cross loadings; top-layer probabilities 0.5). `generate_X_normal`
then samples the latent layers top-down and the Gaussian observations. The truth
(`B_true`, `G_true`) is kept only to score the fits below; with real data only `Y` exists.

The design has $N = 1500$ observations, $J = 18$ variables and $D = 2$ latent layers of
widths $(6, 2)$.

In [2]:
N, K_TRUE, D = 1500, [6, 2], 2
J = 3 * K_TRUE[0]

prop_true, B_true, G_true, gamma_true = build_true_params(K_TRUE, J, 4.0)
prop_true = np.asarray(prop_true, float)
B_true = [np.asarray(B, float) for B in B_true]      # B_true[0]: (J, K1+1); B_true[1]: (K1, K2+1)
G_true = [np.asarray(G, bool) for G in G_true]       # true graphs (nonzero loadings)
gamma_true = np.asarray(gamma_true, float)

np.random.seed(SEED)
Y, A_true = generate_X_normal(N, prop_true, B_true, gamma_true)
print("Y shape:", Y.shape, " value range: [%.2f, %.2f]" % (Y.min(), Y.max()))
print("true widths:", K_TRUE, " true top-layer prop:", prop_true)
print("true observed-layer coefficients B1 (first 6 rows; column 0 = intercept):")
print(B_true[0][:6])

Y shape: (1500, 18)  value range: [-2.66, 3.16]
true widths: [6, 2]  true top-layer prop: [0.5 0.5]
true observed-layer coefficients B1 (first 6 rows; column 0 = intercept):
[[-0.866  1.732  0.     0.     0.     0.     0.   ]
 [-0.866  0.     1.732  0.     0.     0.     0.   ]
 [-0.512  0.     0.     1.977  0.     0.     0.   ]
 [-0.512  0.     0.     0.     1.977  0.     0.   ]
 [-0.743  0.     0.     0.     0.     1.753  0.   ]
 [-0.743  0.     0.     0.     0.     0.     1.753]]


The scoring helper below permutes the fitted latent columns to best match the truth
(latent variables are identified only up to column order), then reports the graph
accuracy (fraction of correctly recovered zero / nonzero loadings) and the coefficient
RMSE of every layer.

In [3]:
def align_to_truth(B_hat_list, B_true_list):
    B_hat = [np.asarray(B, float).copy() for B in B_hat_list]
    for layer, target in enumerate(B_true_list):
        if B_hat[layer].shape != target.shape:
            return None
        cost = np.sum((target[:, 1:].T[:, None, :] - B_hat[layer][:, 1:].T[None, :, :]) ** 2, axis=2)
        true_idx, fit_idx = linear_sum_assignment(cost)
        order = fit_idx[np.argsort(true_idx)]
        B_hat[layer][:, 1:] = B_hat[layer][:, 1:][:, order]
        if layer + 1 < len(B_hat):
            B_hat[layer + 1] = B_hat[layer + 1][order, :]
    return B_hat


def report(result, seconds=None, rmse=True, truth=None):
    truth = B_true if truth is None else truth
    print(f"{result.model}: fitted widths {list(result.K_list)} (truth {K_TRUE})"
          + (f", {seconds:.1f}s" if seconds else ""))
    aligned = align_to_truth(result.B_list, truth)
    if aligned is not None:
        acc = [round(float(np.mean((B[:, 1:] != 0) == (T[:, 1:] != 0))), 3) for B, T in zip(aligned, truth)]
        line = f"  per-layer graph accuracy {acc}"
        if rmse:
            line += f", per-layer coefficient RMSE {[round(float(np.sqrt(np.mean((B - T) ** 2))), 3) for B, T in zip(aligned, truth)]}"
        print(line)

## 3. `fit()` with fixed latent widths

With `dimension_method="fixed"` the widths are supplied through `K_list`. The start reads
the latent states off each latent variable's anchor variables, and penalized stochastic
approximation EM then runs until the relative change of the parameters falls below `tol`
or `max_it` iterations are reached. Hyperparameters that are not given take their
defaults.

In [4]:
t0 = time.perf_counter()
fixed = fit(Y, model="DDE", family="Normal", D=D, K_list=K_TRUE,
            dimension_method="fixed", verbose=False)
report(fixed, seconds=time.perf_counter() - t0)
print("iterations:", fixed.diagnostics["iterations"])
print("defaults filled in:", fixed.diagnostics["default_arguments"])
print("B_list shapes:", [B.shape for B in fixed.B_list], " (column 0 = intercept)")
print("fitted observed-layer coefficients B1 (first 6 rows, aligned to the truth):")
print(np.round(align_to_truth(fixed.B_list, B_true)[0][:6], 2))

DDE: fitted widths [6, 2] (truth [6, 2]), 0.4s
  per-layer graph accuracy [1.0, 1.0], per-layer coefficient RMSE [0.014, 0.368]
iterations: 16
defaults filled in: ['pen_list', 'tau']
B_list shapes: [(18, 7), (6, 3)]  (column 0 = intercept)
fitted observed-layer coefficients B1 (first 6 rows, aligned to the truth):
[[-0.85  1.7   0.    0.    0.    0.    0.  ]
 [-0.88  0.    1.79  0.    0.    0.    0.  ]
 [-0.52  0.    0.    2.    0.    0.    0.  ]
 [-0.51  0.    0.    0.    1.94  0.    0.  ]
 [-0.71  0.    0.    0.    0.    1.73  0.  ]
 [-0.73  0.    0.    0.    0.    0.    1.71]]


## 4. `fit()` with CSP width selection

With `dimension_method="csp"` the widths are not supplied. The fit starts from the maximal
identifiable widths derived from $J$ and lets the cumulative shrinkage process switch
latent columns off; the active widths are returned in `result.K_list` and the starting
widths in `result.diagnostics["K_max_list"]`.

For the 18-variable design the maximal widths are already $(6, 2)$, so to show the
shrinkage we append 12 pure-noise variables: $J = 30$ gives starting widths $(10, 3)$,
and CSP has to recover $(6, 2)$ while the noise variables receive zero loadings.

In [5]:
rng = np.random.default_rng(1)
Y_csp = np.column_stack([Y, rng.normal(size=(N, 12))])   # 18 informative + 12 noise variables

t0 = time.perf_counter()
np.random.seed(SEED)
csp = fit(Y_csp, model="DDE", family="Normal", D=D, dimension_method="csp",
          random_state=SEED, verbose=False)
print("starting widths:", csp.diagnostics["K_max_list"], "-> fitted widths:", list(csp.K_list),
      f"({time.perf_counter() - t0:.1f}s)")
B1_csp = np.asarray(csp.B_list[0])
print("loadings on the 12 noise variables all zero:", bool(np.all(B1_csp[18:, 1:] == 0)))
print("active columns (original indices):", csp.diagnostics["active_columns_original_index"])

starting widths: (10, 3) -> fitted widths: [6, 2] (1.2s)
loadings on the 12 noise variables all zero: True
active columns (original indices): {'layer_1': [1, 2, 3, 4, 5, 9], 'layer_2': [1, 2]}


## 5. Hyperparameter selection

`select_hyperparameters` takes the same arguments as `fit` and a grid of `candidates`, fits
every candidate, scores it, and (with `refit=True`, the default) returns the model refitted
with the best candidate. Other keywords (`K_list`, `max_it`, `verbose`, ...) are passed to
`fit` for every candidate. Scores:

* `criterion="ebic"` (default for DDE and SemiDDE): extended BIC of the full-data fit;
* `"pmse"` (default for CopDDE): predictive synthetic-data utility (propensity-score MSE of
  a random forest separating observed from synthetic data);
* `"bic"` / `"loglik"`: cross-validated likelihood (`n_splits` folds).

Candidates can be given explicitly (axes are crossed as a Cartesian grid; `per_layer` crosses
each layer's values independently). Fixed widths tune `pen_list` and `tau`; CSP tunes
`t_spike`, `tau` and `temp`. The scored table is in `result.diagnostics["cv_table"]` and the
chosen row in `result.raw["selected_hyperparameters"]`.

In [6]:
t0 = time.perf_counter()
selected = select_hyperparameters(
    Y, model="DDE", family="Normal", D=D, dimension_method="fixed", K_list=K_TRUE,
    candidates={"pen_list": {"per_layer": [[N ** 0.25, N ** 0.375], [N ** 0.25]]},
                "tau": [0.2, 0.3, 0.4]},                            # 2 x 1 x 3 = 6 candidates
    random_state=SEED, verbose=False,
)
seconds = time.perf_counter() - t0
table = pd.DataFrame(selected.diagnostics["cv_table"])[["candidate_index", "pen_list", "tau", "ebic", "fit_seconds"]]
table["pen_list"] = table["pen_list"].apply(lambda p: [round(x, 2) for x in p])
best = selected.raw["selected_hyperparameters"]
print(f"{len(table)} candidates scored in {seconds:.1f}s; selected:",
      {k: (round(v, 3) if isinstance(v, float) else [round(x, 2) for x in v]) for k, v in best.items()})
table

6 candidates scored in 2.2s; selected: {'pen_list': [15.53, 6.22], 'tau': 0.4}


,candidate_index,pen_list,tau,ebic,fit_seconds
0,0,"[6.22, 6.22]",0.2,49867.180503,0.334386
1,1,"[6.22, 6.22]",0.3,49868.820894,0.337747
2,2,"[6.22, 6.22]",0.4,49867.695865,0.369179
3,3,"[15.53, 6.22]",0.2,49838.977962,0.371861
4,4,"[15.53, 6.22]",0.3,49840.006134,0.391973
5,5,"[15.53, 6.22]",0.4,49837.867846,0.416251


In [7]:
print("model refitted with the selected hyperparameters:")
report(selected)

model refitted with the selected hyperparameters:
DDE: fitted widths [6, 2] (truth [6, 2])
  per-layer graph accuracy [1.0, 1.0], per-layer coefficient RMSE [0.013, 0.312]


When `candidates` is omitted the package builds its automatic grid: for fixed widths
`auto_grid` crosses layer-specific penalty exponents with threshold choices
($3 \cdot 2^D$ candidates); for DDE and SemiDDE CSP `auto_grid_copula_csp` crosses three
spike scales per layer with the temperatures $\{0.7, 0.9\}$ ($2 \cdot 3^D$ candidates). Each
CSP row of `cv_table` also records the widths that candidate fitted (`fitted_K`).

In [8]:
t0 = time.perf_counter()
auto_fixed = select_hyperparameters(Y, model="DDE", family="Normal", D=D, dimension_method="fixed",
                                    K_list=K_TRUE, random_state=SEED, verbose=False)
best = auto_fixed.raw["selected_hyperparameters"]
print(f"fixed widths, automatic grid: {len(auto_fixed.diagnostics['cv_table'])} candidates in {time.perf_counter() - t0:.1f}s")
print("selected pen_list =", [round(p, 2) for p in best["pen_list"]], " tau =", round(best["tau"], 3))
report(auto_fixed)

fixed widths, automatic grid: 12 candidates in 4.4s
selected pen_list = [15.53, 15.53]  tau = 0.321
DDE: fitted widths [6, 2] (truth [6, 2])
  per-layer graph accuracy [1.0, 1.0], per-layer coefficient RMSE [0.013, 0.334]


In [9]:
t0 = time.perf_counter()
auto_csp = select_hyperparameters(Y_csp, model="DDE", family="Normal", D=D, dimension_method="csp",
                                  random_state=SEED, verbose=False)
best = auto_csp.raw["selected_hyperparameters"]
print(f"CSP, automatic grid: {len(auto_csp.diagnostics['cv_table'])} candidates in {time.perf_counter() - t0:.1f}s")
print("selected t_spike =", best["t_spike"], " temp =", best["temp"], " -> fitted widths", list(auto_csp.K_list), "(truth", K_TRUE, ")")
csp_table = pd.DataFrame(auto_csp.diagnostics["cv_table"])[["candidate_index", "t_spike", "temp", "fitted_K", "ebic"]]
csp_table.sort_values("ebic").head(8)

CSP, automatic grid: 18 candidates in 21.9s
selected t_spike = [0.01, 0.1]  temp = 0.9  -> fitted widths [6, 2] (truth [6, 2] )


,candidate_index,t_spike,temp,fitted_K,ebic
11,11,"[0.01, 0.1]",0.9,"[6, 2]",100992.366094
5,5,"[0.03, 0.1]",0.7,"[6, 2]",100992.753934
15,15,"[0.05, 0.03]",0.9,"[6, 2]",100993.355993
8,8,"[0.05, 0.1]",0.7,"[6, 2]",100993.758884
4,4,"[0.03, 0.05]",0.7,"[6, 2]",100993.978978
17,17,"[0.05, 0.1]",0.9,"[6, 2]",100994.205501
9,9,"[0.01, 0.03]",0.9,"[6, 2]",100997.293528
13,13,"[0.03, 0.05]",0.9,"[6, 2]",100998.249644


## 6. Other models

The calls change only in `model`, `family` and the data. Below, the same latent hierarchy
generates counts (Poisson DDE), log-normal data (SemiDDE) and mixed discrete data (CopDDE). Binary data
carry less information per observation, so the Bernoulli example uses the sparse logistic
design of Lee and Gu (2026), with the same graph and larger logistic coefficients. Each fit uses the package defaults. CopDDE estimates the
first-layer loadings on the scale of its latent Gaussian variables, so only its graph
accuracy is compared with the truth.

In [10]:
def logistic_layer(n_parent):
    """Lee & Gu (2026) design: three children per parent, logit intercepts -2/-4/-2, slopes 4 and 4/3."""
    slopes = np.zeros((3 * n_parent, n_parent))
    slopes[:n_parent] = slopes[n_parent:2 * n_parent] = 4 * np.eye(n_parent)
    slopes[2 * n_parent:] = 4 * np.eye(n_parent)
    for r in range(n_parent):
        slopes[2 * n_parent + r, (r + n_parent // 2) % n_parent] = 4 / 3
    intercept = np.r_[np.full(n_parent, -2.0), np.full(n_parent, -4.0), np.full(n_parent, -2.0)]
    return np.column_stack([intercept, slopes])

B_binary = [logistic_layer(6), logistic_layer(2)]
Y_binary, _, _ = generate_X_bernoulli(N, prop_true, B_binary[0], B_binary[1])
Y_counts, _ = generate_X_poisson(N, prop_true, B_true, random_state=SEED)

np.random.seed(SEED)
t0 = time.perf_counter()
report(fit(Y_binary, model="DDE", family="Bernoulli", D=D, K_list=K_TRUE,
           dimension_method="fixed", verbose=False), seconds=time.perf_counter() - t0, truth=B_binary)
np.random.seed(SEED)
t0 = time.perf_counter()
report(fit(Y_counts, model="DDE", family="Poisson", D=D, K_list=K_TRUE,
           dimension_method="fixed", verbose=False), seconds=time.perf_counter() - t0)

DDE: fitted widths [6, 2] (truth [6, 2]), 0.1s
  per-layer graph accuracy [0.954, 1.0], per-layer coefficient RMSE [0.231, 0.376]


DDE: fitted widths [6, 2] (truth [6, 2]), 0.5s
  per-layer graph accuracy [1.0, 1.0], per-layer coefficient RMSE [0.035, 0.3]


In [11]:
np.random.seed(SEED)
t0 = time.perf_counter()
semi = fit(np.exp(Y), model="SemiDDE", D=D, K_list=K_TRUE, dimension_method="fixed", verbose=False)
report(semi, seconds=time.perf_counter() - t0)
print("  correlation of the estimated pseudo-observations with the latent Gaussian data:",
      round(float(np.corrcoef(semi.transformed[:, 0], Y[:, 0])[0, 1]), 4))

Y_mixed = generate_X_Cop_marg_emp_D(N, prop_true, np.full(J, 3.0), gamma_true, B_true,
                                    random_state=SEED)["X"]   # zero-inflated Poisson, rounded gamma, Poisson
np.random.seed(SEED)
t0 = time.perf_counter()
report(fit(Y_mixed, model="CopDDE", D=D, K_list=K_TRUE, dimension_method="fixed",
           random_state=SEED, verbose=False), seconds=time.perf_counter() - t0, rmse=False)

SemiDDE: fitted widths [6, 2] (truth [6, 2]), 0.6s
  per-layer graph accuracy [1.0, 1.0], per-layer coefficient RMSE [0.015, 0.394]
  correlation of the estimated pseudo-observations with the latent Gaussian data: 0.9992


CopDDE: fitted widths [6, 2] (truth [6, 2]), 9.2s
  per-layer graph accuracy [0.991, 1.0]


## 7. Weights and anchor variables

`plot_weights` draws one heatmap per layer (or a chosen `layer`) of the estimated weights;
`get_anchors` lists, for each latent variable, the observed variables that anchor it, ranked
by the relative loading score $r_{jk} = \max(\min_{l \ne k}(b_{jk} - b_{jl}), 0)$.
`variable_names` labels the observed variables (by default the data frame's column names,
otherwise `y1, y2, ...`). For CSP fits, `latent_feature` is the original (pre-shrinkage)
column index. `dde.visualization.visualization_report` writes a fuller set of plots to disk.

In [12]:
%matplotlib inline
from dde.visualization import plot_weights, get_anchors

names = [f"item{j + 1}" for j in range(J)]
figures = plot_weights(fixed, layer=1, variable_names=names)
get_anchors(fixed, n_anchors=3, variable_names=names).head(9)

,latent_feature,variable,variable_index,rank,score,loading
0,1,item3,3,1,1.995258,1.995258
1,1,item9,9,2,1.972764,1.972764
2,1,item15,15,3,1.789365,1.789365
3,2,item7,7,1,1.714921,1.714921
4,2,item1,1,2,1.704180,1.704180
5,2,item13,13,3,1.672537,1.672537
6,3,item10,10,1,1.989606,1.989606
7,3,item4,4,2,1.942682,1.942682
8,3,item16,16,3,1.844470,1.844470


## Notes

* `tol` bounds the relative change of the parameters for every model. CSP fits average it
  over the last seven iterations and require it in every layer; Bernoulli and Poisson CSP
  fits run their 100 iterations without early stopping.
* Fits draw from NumPy's global random generator: `np.random.seed(...)` before `fit` or
  `select_hyperparameters` reproduces a result. CSP and CopDDE fits also accept
  `random_state`, which seeds their starting values.
* The former keyword `method=` (now `model`) still works with a `DeprecationWarning`, and
  `max_iter` is accepted for `max_it`.